In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

X = train.drop(columns=['id', 'health_condition'])
y = train['health_condition']

In [8]:
for col in X.columns:
    if train[col].isna().any():
        mix = train.groupby(train[col].isna())['health_condition'].value_counts(normalize=True).unstack()
        print(f"===== {col} (missing rate {train[col].isna().mean():.1%}) =====")
        print(mix.round(3), '\n')

===== sleep_duration (missing rate 11.0%) =====
health_condition  at-risk    fit  unhealthy
sleep_duration                             
False               0.859  0.058      0.084
True                0.859  0.057      0.083 

===== heart_rate (missing rate 1.1%) =====
health_condition  at-risk    fit  unhealthy
heart_rate                                 
False               0.859  0.058      0.084
True                0.871  0.055      0.074 

===== bmi (missing rate 2.0%) =====
health_condition  at-risk    fit  unhealthy
bmi                                        
False               0.858  0.057      0.085
True                0.889  0.082      0.029 

===== calorie_expenditure (missing rate 7.7%) =====
health_condition     at-risk    fit  unhealthy
calorie_expenditure                           
False                  0.859  0.058      0.084
True                   0.858  0.058      0.084 

===== step_count (missing rate 2.0%) =====
health_condition  at-risk    fit  unhealthy
step_count

In [9]:
cat_cols = ['stress_level', 'sleep_quality', 'physical_activity_level',
            'smoking_alcohol', 'diet_type', 'gender']
num_cols = [c for c in X.columns if c not in cat_cols]

encode = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                           encoded_missing_value=np.nan), cat_cols),
    ('num', 'passthrough', num_cols),
])

model = Pipeline([
    ('encode', encode),
    ('clf', HistGradientBoostingClassifier(random_state=42)),
])

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(scores.round(5))
print(f"mean: {scores.mean():.5f}  imputed-LR CV was 0.94867, its LB was 0.81601")

[0.96658 0.96687 0.9665  0.96655 0.96637]
mean: 0.96658  imputed-LR CV was 0.94867, its LB was 0.81601


In [11]:
model.fit(X, y)
X_test = test.drop(columns=['id'])
preds = model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'health_condition': preds})
submission.to_csv('../data/submission.csv', index=False)
print(submission['health_condition'].value_counts(normalize=True))

health_condition
at-risk      0.880377
unhealthy    0.069193
fit          0.050431
Name: proportion, dtype: float64
